# PPLSTM order A/B ablation (Kaggle GPU)

Compare **A: mLSTM → PPLSTM → mLSTM → PPLSTM** with **B: PPLSTM → mLSTM → PPLSTM → mLSTM**, keeping the official `xlstm==2.0.6` mLSTM blocks and **the original PPLSTM recurrence and replacement block** from `50m_memory_kaggle.ipynb`.

**This is a runnable experiment, not a result.** Needs Kaggle GPU (T4) and the same FineWeb-Edu Parquet shard as the original benchmarks. The tested architecture in the final two slots is *PPLSTM*, not sLSTM; `slstm_at` is only used to create replacement slots. Two models have the same total number of mLSTM/PPLSTM blocks, same initial weights per *block type* (moved to the other depths), same token batches, optimizer, LR schedule and held-out splits. 10M scored tokens per model by default, one exploratory seed; extend to 3+ seeds before making firm claims. Additional separately trained synthetic memory probes test 128-to-512-token generalization.

Files under `/kaggle/working/pplstm_order_ablation/`: `order_lm_curve.csv`, `order_lm_final.csv`, `order_probe_curves.csv`, `order_probe_final.csv`, and experiment summary ZIP.


In [ ]:
# SETTINGS — change these before Run All.
TARGET_TOKENS = 10_000_000     # scored token presentations, not necessarily distinct tokens
WIDTH, DEPTH, HEADS = 72, 4, 4
CONTEXT, BATCH = 128, 8
SEED = 20261115
TRAIN_GRAPH_REQUIRED = True      # True: do not silently fall back to eager LM training
TRAIN_LM = True
RUN_PROBES = True
PROBE_STEPS = 350               # each of parity, state tracking, delayed recall
PROBE_TRAIN_CONTEXT = 128
PROBE_BATCH = 16
PROBE_EVAL_CONTEXTS = (64, 128, 256, 512)
PROBE_GRAPH_REQUIRED = False     # if probe-specific capture fails, mark eager explicitly
EVAL_EVERY_TOKENS = 2_000_000
SAVE_EVERY_TOKENS = 5_000_000
SMALL_VAL_TOKENS = 8_192
FINAL_VAL_TOKENS = 65_536
STARTUP_SMOKE_ONLY = False      # True: just correctness + mini benchmark (no long training)

# DATA: 'auto', 'prepared', 'fineweb_parquet', or 'tiny_shakespeare'
DATA_MODE = 'fineweb_parquet'
PREPARED_DIR = ''                # directory containing train.bin/valid.bin/test.bin/spm.model
PARQUET_GLOB = '/kaggle/input/**/*.parquet'  # attach a single suitable FineWeb-Edu shard
MAX_PREP_TRAIN_TOKENS = 12_000_000    # reused as necessary to reach 50M presentations
TEXT_FILE = ''                   # optional attached plain UTF-8 file; used in auto mode
RESUME_CHECKPOINT = ''           # optional path to latest.pt from an attached Kaggle output

from pathlib import Path
import os, sys, subprocess, importlib.util
OUT = Path('/kaggle/working/pplstm_order_ablation' if Path('/kaggle/working').exists() else '/mnt/data/pplstm_order_ablation')
OUT.mkdir(parents=True, exist_ok=True)
need=[]
if importlib.util.find_spec('xlstm') is None: need.append('xlstm==2.0.6')
if importlib.util.find_spec('sentencepiece') is None: need.append('sentencepiece')
if importlib.util.find_spec('pyarrow') is None: need.append('pyarrow')
if need:
    print('Installing:', need)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])

import copy, csv, glob, hashlib, json, math, random, statistics, time, traceback, urllib.request, zipfile
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import pandas as pd

assert torch.cuda.is_available(), 'Turn on Kaggle Settings → Accelerator → GPU (T4 x2), then rerun.'
DEVICE=torch.device('cuda:0')
torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print('GPU:',torch.cuda.get_device_name(0),'capability:',torch.cuda.get_device_capability(0))
print('torch:',torch.__version__,'output:',OUT)

def safe_json_write(p, obj):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.pending')
    tmp.write_text(json.dumps(obj,indent=2,default=str))
    os.replace(tmp,p)

def atomic_torch_save(obj,p):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.pending')
    torch.save(obj,tmp)
    os.replace(tmp,p)

def save_csv(p,rows):
    if rows: pd.DataFrame(rows).to_csv(p,index=False)

def export_metric_bundle():
    # Refresh after every milestone; the results survive even if later training stops.
    archive=OUT/'PPLSTM_Kaggle_50M_Memory_Results.zip'
    temporary=OUT/(archive.name+'.pending')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for file in list(OUT.glob('*.csv'))+list(OUT.glob('*.json')):
            z.write(file,arcname=file.name)
    os.replace(temporary,archive)
    return archive

def params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)
ABLATION_SEEDS = [SEED]  # Increase to >=3 for a robust answer
PROBE_SEEDS = [5002]   # Increase to >=3 for a robust answer


## Original PPLSTM cell (unaltered)

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Adapted from the original PPLSTM project by Rickey (https://github.com/rickey1990/PPLSTM/blob/main/pplstm.py)
# The transition equations and scan implementation are kept as published.
"""PPLSTM: associative 3-channel permutation-mixed recurrent cell."""
import itertools
import math
import torch
from torch import Tensor, nn
from torch.nn import functional as F


def shift_right(x: Tensor) -> Tensor:
    return torch.cat((torch.zeros_like(x[:, :1]), x[:, :-1]), dim=1)


def scalar_affine_scan(a: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    A, B = a, b
    step = 1
    while step < a.shape[1]:
        A, B = (
            torch.cat((A[:, :step], A[:, step:] * A[:, :-step]), dim=1),
            torch.cat((B[:, :step], B[:, step:] + A[:, step:] * B[:, :-step]), dim=1),
        )
        step *= 2
    return A, B


def matrix_affine_scan(A: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    step = 1
    while step < A.shape[1]:
        next_A = torch.cat((A[:, :step], A[:, step:] @ A[:, :-step]), dim=1)
        next_b = torch.cat((
            b[:, :step],
            b[:, step:] + (A[:, step:] @ b[:, :-step].unsqueeze(-1)).squeeze(-1),
        ), dim=1)
        A, b = next_A, next_b
        step *= 2
    return A, b


class PPLSTM(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        if width < 3 or width % 3 != 0:
            raise ValueError('width must be a positive multiple of 3')
        self.width = width
        self.groups = width // 3
        self.xproj = nn.Linear(width, 4 * width)
        self.mix_logit = nn.Parameter(torch.full((4, width), math.log(3.0)))
        self.sign_proj = nn.Linear(width, width)
        self.initial_cell = nn.Parameter(torch.linspace(-1.5, 1.5, 3).repeat(self.groups))
        self.input_bias = nn.Parameter(torch.full((width,), -4.0))
        self.forget_bias = nn.Parameter(torch.full((width,), 4.0))
        nn.init.zeros_(self.sign_proj.weight)
        nn.init.constant_(self.sign_proj.bias, 2.8)

        bank = []
        for order in itertools.permutations(range(3)):
            matrix = torch.zeros(3, 3)
            for row, column in enumerate(order):
                matrix[row, column] = 1.0
            bank.append(matrix)
        self.register_buffer('permutation_bank', torch.stack(bank))

        self.selector = nn.Linear(width, self.groups * 6)
        nn.init.zeros_(self.selector.weight)
        nn.init.constant_(self.selector.bias, -1.0)
        with torch.no_grad():
            for group in range(self.groups):
                self.selector.bias[group * 6] = 3.0

    def _token_maps(self, x: Tensor):
        if x.ndim != 3 or x.shape[-1] != self.width:
            raise ValueError('Expected x with shape [batch, time, width]')
        batch, length, width = x.shape
        group_count = self.groups
        previous = shift_right(x)
        coeff = self.mix_logit.sigmoid().view(1, 1, 4, width)
        gates = (self.xproj(x).reshape(batch, length, 4, width) * coeff
                 + self.xproj(previous).reshape(batch, length, 4, width) * (1 - coeff))
        input_logits, forget_logits, candidate_logits, output_logits = gates.unbind(2)
        mixed = x * self.mix_logit[0].sigmoid() + previous * (1 - self.mix_logit[0].sigmoid())
        signs = self.sign_proj(mixed).tanh().reshape(batch, length, group_count, 3)
        z = candidate_logits.tanh().reshape(batch, length, group_count, 3)
        log_i = (input_logits + self.input_bias).reshape(batch, length, group_count, 3).mean(-1)
        log_f = F.logsigmoid((forget_logits + self.forget_bias)
                           .reshape(batch, length, group_count, 3).mean(-1))
        f_prefix = log_f.cumsum(dim=1)
        m = f_prefix + torch.cummax(torch.cat((
            torch.zeros_like(f_prefix[:, :1]), log_i - f_prefix,
        ), dim=1), dim=1).values[:, 1:]
        f = (log_f + shift_right(m) - m).exp()
        i = (log_i - m).exp()
        weights = self.selector(mixed).reshape(batch, length, group_count, 6).softmax(-1)
        P = torch.einsum('btgp,pij->btgij', weights, self.permutation_bank)
        A = f[..., None, None] * P * signs[..., None, :]
        b = i[..., None] * z
        return A, b, f, i, output_logits

    def forward(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        prefix_A, prefix_b = matrix_affine_scan(A, b)
        c0 = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        c = (prefix_A @ c0[:, None, :, :, None]).squeeze(-1) + prefix_b
        norm_A, norm_b = scalar_affine_scan(f, i)
        n = norm_A + norm_b
        ratio = (c / n[..., None].clamp_min(1e-12)).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

    def serial_reference(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        c = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        n = x.new_ones(batch, self.groups)
        states = []
        for t in range(length):
            c = (A[:, t] @ c.unsqueeze(-1)).squeeze(-1) + b[:, t]
            n = f[:, t] * n + i[:, t]
            states.append(c / n[..., None].clamp_min(1e-12))
        ratio = torch.stack(states, dim=1).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

## Two xLSTM block arrangements

In [ ]:
from xlstm import (xLSTMBlockStack, xLSTMBlockStackConfig, mLSTMBlockConfig,
                   mLSTMLayerConfig, sLSTMBlockConfig, sLSTMLayerConfig, FeedForwardConfig)

class PPLSTMReplacementLayer(nn.Module):
    """Adapt original sLSTM layer slot; retain convolution, group norm and dropout."""
    def __init__(self, original_layer):
        super().__init__()
        self.config = original_layer.config
        self.conv1d = original_layer.conv1d if self.config.conv1d_kernel_size > 0 else None
        self.conv_act_fn = original_layer.conv_act_fn if self.conv1d is not None else None
        self.group_norm = original_layer.group_norm
        self.dropout = original_layer.dropout
        self.pplstm = PPLSTM(self.config.embedding_dim)

    def forward(self, x, **kwargs):
        B,S,W=x.shape
        conv_x=self.conv_act_fn(self.conv1d(x)) if self.conv1d is not None else x
        y=self.dropout(self.pplstm(conv_x))
        H=self.config.num_heads
        y=y.reshape(B,S,H,W//H).transpose(1,2)
        y=self.group_norm(y).transpose(1,2).reshape(B,S,W)
        return y


def build_backbone(width=WIDTH, context=CONTEXT, order='mlstm_first'):
    assert order in ('mlstm_first','pplstm_first')
    pplstm_at = [1,3] if order=='mlstm_first' else [0,2]
    expected_map = [0,1,0,1] if order=='mlstm_first' else [1,0,1,0]
    assert width%3==0 and width%HEADS==0
    cfg=xLSTMBlockStackConfig(
        mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig(
            conv1d_kernel_size=4, qkv_proj_blocksize=4, num_heads=HEADS)),
        slstm_block=sLSTMBlockConfig(slstm=sLSTMLayerConfig(
            backend='vanilla', num_heads=HEADS, conv1d_kernel_size=4,
            bias_init='powerlaw_blockdependent'),
            feedforward=FeedForwardConfig(proj_factor=1.3, act_fn='gelu')),
        context_length=context, num_blocks=DEPTH,
        embedding_dim=width, slstm_at=pplstm_at)
    stack=xLSTMBlockStack(cfg)
    assert list(stack.config.block_map)==expected_map
    for index in pplstm_at
        original=stack.blocks[index].xlstm
        if not hasattr(original,'slstm_cell'):
            raise RuntimeError('xLSTM API changed: expected slstm_cell in layer '+str(index))
        stack.blocks[index].xlstm=PPLSTMReplacementLayer(original)
    return stack

class PPLSTMLanguageModel(nn.Module):
    def __init__(self, vocab, width=WIDTH, context=CONTEXT, order='mlstm_first'):
        super().__init__()
        self.vocab=vocab
        self.emb=nn.Embedding(vocab,width)
        nn.init.normal_(self.emb.weight,std=0.02)
        self.backbone=build_backbone(width,context,order)
        self.norm=nn.LayerNorm(width)
        self.out_bias=nn.Parameter(torch.zeros(vocab))
    def forward(self, tokens):
        h=self.norm(self.backbone(self.emb(tokens)))
        return F.linear(h,self.emb.weight,self.out_bias)  # tied LM output weights

class PPLSTMProbeModel(nn.Module):
    """Same four-block backbone, dedicated embedding & final-token classifier."""
    def __init__(self,num_classes,width=WIDTH,max_context=max(PROBE_EVAL_CONTEXTS),vocab=16,order='mlstm_first'):
        super().__init__()
        self.emb=nn.Embedding(vocab,width)
        self.backbone=build_backbone(width,max_context,order)
        self.norm=nn.LayerNorm(width)
        self.head=nn.Linear(width,num_classes)
    def forward(self,tokens):
        return self.head(self.norm(self.backbone(self.emb(tokens))[:,-1]))

model_demo=PPLSTMLanguageModel(256).to(DEVICE)
print('4-block model structure:',[type(model_demo.backbone.blocks[i].xlstm).__name__ for i in range(4)])
print('Parameter count at vocab 256:',params(model_demo))
del model_demo
torch.cuda.empty_cache()

## PPLSTM scan and default-stack smoke tests

In [ ]:
DIAGNOSTICS=[]
def record_check(name, passed, **data):
    row={'test':name,'passed':bool(passed),**data}
    DIAGNOSTICS.append(row)
    print(('PASS' if passed else 'FAIL'),name, {k:v for k,v in data.items() if k not in ('traceback',)})
    save_csv(OUT/'diagnostics.csv',DIAGNOSTICS)
    return row

def check_pplstm_scan(width=12,length=17,precision=torch.float64):
    torch.manual_seed(57)
    cell=PPLSTM(width).to(device=DEVICE,dtype=precision)
    x=torch.randn(2,length,width,device=DEVICE,dtype=precision,requires_grad=True)
    parallel=cell(x); serial=cell.serial_reference(x)
    err=(parallel-serial).abs().max().item()
    pa=torch.autograd.grad(parallel.square().sum(),(x,*cell.parameters()),retain_graph=True)
    sa=torch.autograd.grad(serial.square().sum(),(x,*cell.parameters()))
    grad_err=max((a-b).abs().max().item() for a,b in zip(pa,sa))
    tol=1e-9 if precision==torch.float64 else 1e-4
    record_check('parallel_vs_serial_scan_'+str(precision).split('.')[-1],
                 err<tol and grad_err<tol*10,output_max_error=err,grad_max_error=grad_err,length=length)
    assert DIAGNOSTICS[-1]['passed'],'Parallel recurrence numerical check failed'

check_pplstm_scan(12,17,torch.float64)
check_pplstm_scan(48,128,torch.float32)

# A causal model must not change outputs at earlier positions if later token IDs change.
torch.manual_seed(91)
causal_model=PPLSTMLanguageModel(256).to(DEVICE).eval()
a=torch.randint(0,256,(2,48),device=DEVICE)
b=a.clone(); b[:,24:]=torch.randint(0,256,(2,24),device=DEVICE)
with torch.no_grad():
    ya=causal_model(a)[:,:24]; yb=causal_model(b)[:,:24]
causal_error=(ya-yb).abs().max().item()
record_check('full_stack_future_token_causality',causal_error<1e-5,max_error=causal_error)
assert DIAGNOSTICS[-1]['passed'],'Full model failed causality test'
del causal_model,a,b,ya,yb

torch.manual_seed(92)
finite_model=PPLSTMLanguageModel(256).to(DEVICE)
x=torch.randint(0,256,(2,128),device=DEVICE)
target=torch.randint(0,256,(2,128),device=DEVICE)
loss=F.cross_entropy(finite_model(x).reshape(-1,256),target.reshape(-1))
loss.backward()
all_finite=bool(torch.isfinite(loss).item()) and all(p.grad is not None and bool(torch.isfinite(p.grad).all().item()) for p in finite_model.parameters())
record_check('full_stack_finite_loss_and_gradients',all_finite,loss=float(loss.item()),parameters=params(finite_model))
assert all_finite,'Full model produced missing or nonfinite gradients'
del finite_model,x,target,loss
torch.cuda.empty_cache()

## Original CUDA Graph training engine, without standalone graph/eager benchmark

In [ ]:
# CUDA Graph records forward+backward on fixed [batch, context] tensors.
# AdamW step and clipping remain outside capture. This is CUDA Graph replay,
# NOT a new fused CUDA kernel; the PPLSTM scan is the same as above.

def task_loss(logits,labels):
    if logits.ndim==3: return F.cross_entropy(logits.reshape(-1,logits.shape[-1]),labels.reshape(-1),ignore_index=-100)
    return F.cross_entropy(logits,labels)

class CapturedForwardBackward:
    def __init__(self,model,example_x,example_y):
        assert example_x.device.type=='cuda'
        self.model=model
        self.x=torch.zeros_like(example_x)
        self.y=torch.zeros_like(example_y)
        self.graph=torch.cuda.CUDAGraph()
        self.model.train()
        side=torch.cuda.Stream(device=example_x.device)
        side.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(side):
            for _ in range(3):
                self.model.zero_grad(set_to_none=True)
                task_loss(self.model(self.x),self.y).backward()
        torch.cuda.current_stream().wait_stream(side)
        self.model.zero_grad(set_to_none=True)
        with torch.cuda.graph(self.graph,stream=side):
            self.loss=task_loss(self.model(self.x),self.y)
            self.loss.backward()
        # Keep graph-owned gradient buffers alive; don't set grads to None.

    def replay(self,x,y):
        if x.shape!=self.x.shape or y.shape!=self.y.shape:
            raise ValueError('CUDA Graph shape mismatch')
        self.x.copy_(x); self.y.copy_(y)
        self.graph.replay()
        return self.loss


def update_graph_step(captured,optimizer,x,y,lr=None):
    if lr is not None:
        for group in optimizer.param_groups: group['lr']=float(lr)
    loss=captured.replay(x,y)
    torch.nn.utils.clip_grad_norm_(captured.model.parameters(),1.0)
    optimizer.step()
    return loss


def update_eager_step(model,optimizer,x,y,lr=None):
    if lr is not None:
        for group in optimizer.param_groups: group['lr']=float(lr)
    optimizer.zero_grad(set_to_none=True)
    loss=task_loss(model(x),y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    optimizer.step()
    return loss




## FineWeb-Edu preprocessing (existing experiment)

In [ ]:
"""Frozen FineWeb-Edu sample-10BT data pipeline.
Only the dataset/test *types* were taken from the provided example notebook.
No model code or experiment results from that notebook are imported.
"""
import glob,hashlib,json,os,time
from pathlib import Path
import numpy as np


def sha_text(x):
    return hashlib.sha256(x.encode('utf8','ignore')).hexdigest()


def split_doc(text):
    """Exact duplicates land in the same split; near-duplicates remain a caveat."""
    r=int(sha_text(text)[:12],16)%1000
    return 'valid' if r<10 else 'test' if r<20 else 'train'


def documents(paths):
    import pyarrow.parquet as pq
    for filename in paths:
        pqfile=pq.ParquetFile(str(filename))
        if 'text' not in pqfile.schema.names:
            raise ValueError(f'Not a FineWeb text Parquet shard: {filename}')
        for batch in pqfile.iter_batches(columns=['text'],batch_size=512):
            for raw in batch.column(0).to_pylist():
                if isinstance(raw,str) and len(raw.strip())>=20:
                    yield raw.replace('\x00',' ').strip()


def find_paths(pattern):
    found=[Path(p) for p in sorted(glob.glob(pattern,recursive=True)) if Path(p).is_file()]
    if not found:
        raise FileNotFoundError('FineWeb-Edu parquet not found. Attach sample/10BT/000_00000.parquet in Kaggle Inputs, then edit PARQUET_GLOB. Current: '+pattern)
    if len(found)>1:
        filtered=[p for p in found if 'fineweb' in str(p).lower()]
        if filtered:found=filtered
        else:raise RuntimeError('Multiple generic parquet shards matched. Narrow PARQUET_GLOB to FineWeb-Edu sample-10BT only.')
    return found


def manifest_for_input(paths):
    rows=[]
    for p in paths:
        with p.open('rb') as stream:head=stream.read(1024*1024)
        rows.append({'path':str(p),'file_size':p.stat().st_size,
             'first_megabyte_sha256':hashlib.sha256(head).hexdigest()})
    return rows


def prepare(root,glob_pattern,existing_data_root='',vocab=4096,max_train=30_000_000,
            valid_count=250_000,test_count=250_000,sample_chars=3_000_000,
            stop_epoch=None):
    import sentencepiece as spm
    out=Path(root)/'data';out.mkdir(parents=True,exist_ok=True)
    if existing_data_root:
        base=Path(existing_data_root)/'data'
        if not (base/'spm.model').exists() or not (base/'train.bin').exists():
            raise ValueError('EXISTING_DATA_ROOT must contain data/spm.model and train/valid/test.bin')
        manifest=json.loads((base/'manifest.json').read_text()) if (base/'manifest.json').exists() else {}
        return {'root':str(base),'tokenizer':str(base/'spm.model'),'sizes':{
            part:(base/f'{part}.bin').stat().st_size//2 for part in ('train','valid','test')},
            'reused':True,'original_manifest':manifest,'source':'FineWeb-Edu, externally prepared corpus (verify uploaded provenance)'}

    old=out/'manifest.json'
    if old.exists() and (out/'spm.model').exists():
        manifest=json.loads(old.read_text())
        if manifest.get('complete') and all((out/f'{k}.bin').exists() for k in ('train','valid','test')):
            return manifest
    paths=find_paths(glob_pattern)
    source_manifest=manifest_for_input(paths)
    sample=out/'spm_training_sample.txt'
    if not (out/'spm.model').exists():
        chars=0;count=0
        with sample.open('w',encoding='utf8') as f:
            for t in documents(paths):
                if split_doc(t)!='train':continue
                t=' '.join(t[:15000].split());f.write(t+'\n');chars+=len(t);count+=1
                if chars>=sample_chars:break
        if chars<20000:raise RuntimeError('Not enough training text to fit a tokenizer')
        spm.SentencePieceTrainer.Train(input=str(sample),model_prefix=str(out/'spm'),
            vocab_size=vocab,model_type='bpe',character_coverage=0.9995,
            unk_id=0,eos_id=1,bos_id=-1,pad_id=-1,
            max_sentence_length=16384,hard_vocab_limit=False, num_threads=4)
        sample.unlink(missing_ok=True)
    sp=spm.SentencePieceProcessor(model_file=str(out/'spm.model'))
    if sp.vocab_size()>65535:raise ValueError('uint16 token array requires vocab<=65535')
    sizes={'train':int(max_train),'valid':int(valid_count),'test':int(test_count)}
    mm={}
    for k,n in sizes.items():
        mm[k]=np.memmap(out/f'{k}.bin',dtype=np.uint16,mode='w+',shape=(n,))
    pos={k:0 for k in sizes}
    doc_count={k:0 for k in sizes}
    print('Tokenizing attached FineWeb-Edu. Caps:',sizes,flush=True)
    for j,t in enumerate(documents(paths)):
        if stop_epoch is not None and time.time()>stop_epoch:
            print('Corpus preparation budget reached; using filled prefixes, if sufficient.',flush=True)
            break
        split=split_doc(t)
        remain=sizes[split]-pos[split]
        if remain<=0:
            if all(pos[k]>=sizes[k] for k in sizes):break
            continue
        ids=sp.encode(t,out_type=int)
        if not ids:continue
        ids.append(sp.eos_id())
        take=min(len(ids),remain)
        mm[split][pos[split]:pos[split]+take]=np.asarray(ids[:take],dtype=np.uint16)
        pos[split]+=take;doc_count[split]+=1
        if j and j%5000==0:print('data',j,'documents',pos,flush=True)
    for k in sizes:
        mm[k].flush();del mm[k]
        with (out/f'{k}.bin').open('r+b') as f:f.truncate(pos[k]*2)
    if pos['train']<512*128 or min(pos['valid'],pos['test'])<4096:
        raise RuntimeError('Not enough authentic FineWeb-Edu text before preparation deadline: '+str(pos))
    manifest={'complete':True,'root':str(out),'tokenizer':str(out/'spm.model'),
        'dataset':'HuggingFaceFW/fineweb-edu sample-10BT','data_provenance':source_manifest,
        'document_hash_split':'1% validation, 1% test, 98% train; exact hash by full text',
        'document_counts':doc_count,'sizes':pos,'vocab_size':sp.vocab_size(),
        'reused':False,'tokens_are_bpe':True,
        'limits':sizes,'sha256_tokenizer':hashlib.sha256((out/'spm.model').read_bytes()).hexdigest()}
    old.write_text(json.dumps(manifest,indent=2))
    return manifest


def load_tokens(manifest,part):
    f=Path(manifest['root'])/f'{part}.bin'
    return np.memmap(f,mode='r',dtype=np.uint16)

In [ ]:
from pathlib import Path

def available_prepared_dirs():
    bases=[]
    if PREPARED_DIR: bases.append(Path(PREPARED_DIR))
    for m in glob.glob('/kaggle/input/**/train.bin',recursive=True): bases.append(Path(m).parent)
    # Also reuse local data produced in an earlier run of the same notebook.
    bases.append(OUT/'data')
    return list(dict.fromkeys(bases))

def text_to_bytes_file(text):
    # Byte tokenizer avoids relying on an externally stored vocabulary for fallback.
    arr=np.frombuffer(text.encode('utf-8'),dtype=np.uint8).astype(np.uint16)
    t=int(len(arr)*0.90);v=int(len(arr)*0.95)
    if min(v-t,len(arr)-v)<8192: raise ValueError('Text file too short for validation and test')
    return {'train':arr[:t].copy(),'valid':arr[t:v].copy(),'test':arr[v:].copy()}, 256

def load_data():
    if DATA_MODE in ('auto','prepared'):
        for d in available_prepared_dirs():
            if all((d/x).exists() for x in ['train.bin','valid.bin','test.bin','spm.model']):
                import sentencepiece as spm
                sp=spm.SentencePieceProcessor(model_file=str(d/'spm.model'))
                arrays={split:np.memmap(d/(split+'.bin'),dtype=np.uint16,mode='r') for split in ('train','valid','test')}
                result={'mode':'prepared_fineweb','root':str(d),'vocab':sp.vocab_size(),'lengths':{k:len(v) for k,v in arrays.items()}}
                return arrays,sp.vocab_size(),result
        if DATA_MODE=='prepared':raise FileNotFoundError('Set PREPARED_DIR to directory containing .bin files + spm.model')
    if DATA_MODE in ('auto','fineweb_parquet'):
        paths=glob.glob(PARQUET_GLOB,recursive=True)
        if paths:
            # Reuse EXACT project's document hash split and SentencePiece preprocessing.
            if len(paths)>1:
                fw=[s for s in paths if 'fineweb' in s.lower() or 'cartman' in s.lower()]
                if len(fw)==1: paths=fw
                else:raise RuntimeError('More than one parquet found. Narrow PARQUET_GLOB to one FineWeb shard.')
            manifest=prepare(str(OUT),paths[0],vocab=4096,
                             max_train=MAX_PREP_TRAIN_TOKENS,valid_count=150000,test_count=150000)
            arrays={split:load_tokens(manifest,split) for split in ('train','valid','test')}
            return arrays,int(manifest['vocab_size']),{'mode':'fineweb_parquet','manifest':manifest,'lengths':{k:len(v) for k,v in arrays.items()}}
        if DATA_MODE=='fineweb_parquet':raise FileNotFoundError('Attach a FineWeb-Edu parquet via Add Input')
    path=Path(TEXT_FILE) if TEXT_FILE else None
    if path and path.exists():
        raw=path.read_text(encoding='utf-8')
        arrays,vocab=text_to_bytes_file(raw)
        return arrays,vocab,{'mode':'attached_text_byte_tokenizer','path':str(path),'lengths':{k:len(v) for k,v in arrays.items()}}
    if DATA_MODE not in ('auto','tiny_shakespeare'):
        raise RuntimeError('Unknown DATA_MODE: '+DATA_MODE)
    print('WARNING: No prepared FineWeb input. Using much smaller Tiny Shakespeare text, repeatedly sampled.')
    cache=OUT/'tiny_shakespeare.txt'
    if not cache.exists():
        urllib.request.urlretrieve('https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',cache)
    arrays,vocab=text_to_bytes_file(cache.read_text(encoding='utf-8'))
    return arrays,vocab,{'mode':'tiny_shakespeare_byte_tokenizer','path':str(cache),'lengths':{k:len(v) for k,v in arrays.items()},
                        'warning':'50M tokens are presentations with heavy reuse, NOT 50M distinct tokens'}

TOKENS,VOCAB,DATA_INFO=load_data()
assert min(map(len,TOKENS.values()))>=CONTEXT+8192
assert max(int(np.max(a)) for a in TOKENS.values())<VOCAB
DATA_INFO['train_sha256_first_1mb']=hashlib.sha256(np.asarray(TOKENS['train'][:524288],dtype=np.uint16).tobytes()).hexdigest()
safe_json_write(OUT/'data_info.json',DATA_INFO)
print('Dataset:',DATA_INFO)
print('Model vocabulary:',VOCAB)

## Deterministic language batches, evaluation and learning-rate schedule

In [ ]:
def make_lm_batch(step,valid_tokens=CONTEXT*BATCH):
    train=TOKENS['train']
    max_window=(len(train)-CONTEXT-1)//CONTEXT
    rng=np.random.default_rng(SEED+1_000_003*int(step))
    picks=rng.integers(0,max_window,size=BATCH)
    w=np.stack([np.asarray(train[int(i)*CONTEXT:int(i)*CONTEXT+CONTEXT+1],dtype=np.int64) for i in picks])
    x=torch.from_numpy(w[:,:-1].copy()).to(DEVICE)
    y=torch.from_numpy(w[:,1:].copy()).to(DEVICE)
    if valid_tokens<CONTEXT*BATCH:
        y.reshape(-1)[valid_tokens:]=-100  # exact scored-token budget on final batch
    return x,y

@torch.no_grad()
def eval_lm(model,split='valid',max_tokens=SMALL_VAL_TOKENS):
    model.eval()
    arr=TOKENS[split]; windows=min(max_tokens//CONTEXT,(len(arr)-1)//CONTEXT)
    if windows<1:raise RuntimeError('Not enough validation text')
    nll=0.; right=0; seen=0
    for start in range(0,windows,BATCH):
        bs=min(BATCH,windows-start)
        w=np.stack([np.asarray(arr[(start+i)*CONTEXT:(start+i)*CONTEXT+CONTEXT+1],dtype=np.int64) for i in range(bs)])
        x=torch.from_numpy(w[:,:-1].copy()).to(DEVICE)
        y=torch.from_numpy(w[:,1:].copy()).to(DEVICE)
        logits=model(x)
        nll+=float(F.cross_entropy(logits.reshape(-1,VOCAB),y.reshape(-1),reduction='sum').item())
        right+=int((logits.argmax(-1)==y).sum().item())
        seen+=int(y.numel())
    model.train()
    loss=nll/seen
    return {'tokens':seen,'nll':loss,'perplexity':math.exp(loss),'top1_accuracy_pct':100*right/seen}


def lr_schedule(tokens_seen):
    warm=1_000_000; peak=8e-4; floor=8e-5
    if tokens_seen<warm:return peak*max(tokens_seen+CONTEXT,1)/warm
    frac=min(1.,max(0.,(tokens_seen-warm)/max(TARGET_TOKENS-warm,1)))
    return floor+(peak-floor)*0.5*(1+math.cos(math.pi*frac))



## Language A/B, matched weights and fixed data

In [ ]:

# Order A and B share initial parameters *by block type*, not merely random seed.
import re, gc
ORDERS=('mlstm_first','pplstm_first')
def count_params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

def make_pair(factory,seed):
    torch.manual_seed(int(seed)); a=factory(ORDERS[0])
    torch.manual_seed(int(seed)+1); b=factory(ORDERS[1])
    assert count_params(a)==count_params(b)
    sa=a.state_dict(); sb=b.state_dict()
    moved={0:1,1:0,2:3,3:2}   # destination B block : source A block
    for key in sb:
        match=re.match(r'^backbone\.blocks\.(\d+)\.(.*)$',key)
        old=(f'backbone.blocks.{moved[int(match.group(1))]}.{match.group(2)}' if match else key)
        assert old in sa and sa[old].shape==sb[key].shape, (key,old)
        sb[key]=sa[old].detach().clone()
    b.load_state_dict(sb)
    assert all(torch.equal(v,sa[(f'backbone.blocks.{moved[int(mt.group(1))]}.{mt.group(2)}' if (mt:=re.match(r'^backbone\.blocks\.(\d+)\.(.*)$',k)) else k)]) for k,v in b.state_dict().items())
    return a,b

def snapshot_pair(factory,seed):
    a,b=make_pair(factory,seed)
    out={ORDERS[0]:{k:v.detach().cpu().clone() for k,v in a.state_dict().items()},
         ORDERS[1]:{k:v.detach().cpu().clone() for k,v in b.state_dict().items()}}
    print('Equal parameter counts:',count_params(a),count_params(b))
    del a,b
    return out

def start_backend(model,x,y):
    try: return 'cuda_graphs',CapturedForwardBackward(model,x,y)
    except Exception as e:
        if TRAIN_GRAPH_REQUIRED: raise RuntimeError('CUDA Graphs failed; no hidden eager fallback') from e
        print('Explicit eager fallback:',repr(e))
        return 'eager',None

def update(model,opt,backend,graph,x,y,lr):
    return update_graph_step(graph,opt,x,y,lr) if backend=='cuda_graphs' else update_eager_step(model,opt,x,y,lr)

def causal_check(model):
    model.eval()
    a=torch.randint(0,VOCAB,(2,32),device=DEVICE)
    b=a.clone(); b[:,16:]=torch.randint(0,VOCAB,(2,16),device=DEVICE)
    with torch.no_grad(): error=(model(a)[:,:16]-model(b)[:,:16]).abs().max().item()
    model.train()
    assert error<5e-5, ('Causality failed',error)
    return error

LM_ROWS=[];LM_FINAL=[]
for seed in ABLATION_SEEDS:
    states=snapshot_pair(lambda order:PPLSTMLanguageModel(VOCAB,order=order),seed)
    for order in ORDERS:
        model=PPLSTMLanguageModel(VOCAB,order=order).to(DEVICE)
        model.load_state_dict(states[order])
        causal_error=causal_check(model)
        opt=torch.optim.AdamW(model.parameters(),lr=8e-4,betas=(.9,.95),weight_decay=.01)
        x0,y0=make_lm_batch(0)
        backend,graph=start_backend(model,x0,y0)
        print('Training',order,'seed',seed,'parameters',count_params(model),'backend',backend)
        elapsed=0.;seen=0;step=0
        val=eval_lm(model,'valid',SMALL_VAL_TOKENS)
        LM_ROWS.append(dict(seed=seed,order=order,scored_tokens=0,steps=0,
                            val_ppl=val['perplexity'],val_nll=val['nll'],backend=backend))
        start=time.perf_counter()
        while seen<TARGET_TOKENS:
            count=min(CONTEXT*BATCH,TARGET_TOKENS-seen)
            x,y=make_lm_batch(step,valid_tokens=count)  # exactly same seed/step -> same data
            lr=lr_schedule(seen)
            loss=update(model,opt,backend,graph,x,y,lr)
            step+=1;seen+=count
            if seen//EVAL_EVERY_TOKENS!=(seen-count)//EVAL_EVERY_TOKENS or seen==TARGET_TOKENS:
                torch.cuda.synchronize();elapsed+=time.perf_counter()-start
                val=eval_lm(model,'valid',SMALL_VAL_TOKENS)
                rec=dict(seed=seed,order=order,scored_tokens=seen,steps=step,train_loss=loss.item(),
                         val_ppl=val['perplexity'],val_nll=val['nll'],
                         training_seconds=elapsed,tokens_per_second=seen/max(elapsed,1e-9),
                         parameter_count=count_params(model),backend=backend)
                LM_ROWS.append(rec);save_csv(OUT/'order_lm_curve.csv',LM_ROWS)
                print(rec)
                start=time.perf_counter()
        test=eval_lm(model,'test',FINAL_VAL_TOKENS)
        val=eval_lm(model,'valid',FINAL_VAL_TOKENS)
        rec=dict(seed=seed,order=order,train_tokens=seen,parameter_count=count_params(model),
                 val_ppl=val['perplexity'],test_ppl=test['perplexity'],
                 test_nll=test['nll'],test_top1_accuracy_pct=test['top1_accuracy_pct'],
                 tokens_per_second=seen/max(elapsed,1e-9),backend=backend)
        LM_FINAL.append(rec);save_csv(OUT/'order_lm_final.csv',LM_FINAL)
        print('FINAL',rec)
        del model,opt,graph
        torch.cuda.empty_cache();gc.collect()


## Separate supervised memory benchmarks (not a test of language-model weights)

In [ ]:
PROBE_ROWS=[];PROBE_CURVES=[]
def synthetic_batch(task,batch,length,seed):
    rng=np.random.default_rng(int(seed))
    if task=='parity':
        bits=rng.integers(0,2,size=(batch,length-1),dtype=np.int64)
        x=np.concatenate([bits,np.full((batch,1),3,dtype=np.int64)],axis=1)
        labels=(bits.sum(axis=1)%2).astype(np.int64)
    elif task=='state_mod4':
        # Stratify candidate examples to give balanced labels (25% chance baseline).
        # Reset is retained in the task; without balancing, state 0 would be overrepresented.
        candidate_count=max(batch*8,64)
        operations=rng.choice(np.array([4,5,6,7],dtype=np.int64),
                              size=(candidate_count,length-1),p=[.3,.3,.3,.1])
        indices=np.arange(length-1)[None,:]
        last_reset=np.where(operations==7,indices,-1).max(axis=1)
        active=indices>last_reset[:,None]
        candidate_labels=((((operations==4)&active).sum(axis=1)-
                          ((operations==5)&active).sum(axis=1))%4).astype(np.int64)
        selected=[]
        for klass in range(4):
            available=np.flatnonzero(candidate_labels==klass)
            need=batch//4+(1 if klass<batch%4 else 0)
            selected.extend(rng.choice(available,size=need,replace=len(available)<need).tolist())
        rng.shuffle(selected)
        labels=candidate_labels[selected]
        chosen=operations[selected]
        x=np.concatenate([chosen,np.full((batch,1),8,dtype=np.int64)],axis=1)
    elif task=='delayed_recall':
        symbols=rng.integers(9,15,size=(batch,length-1),dtype=np.int64)
        labels=(symbols[:,0]-9).astype(np.int64)
        x=np.concatenate([symbols,np.full((batch,1),15,dtype=np.int64)],axis=1)
    else:raise ValueError(task)
    return torch.from_numpy(x).to(DEVICE),torch.from_numpy(labels.astype(np.int64)).to(DEVICE)

@torch.no_grad()
def eval_probe(model,task,length,batches=12):
    model.eval()
    correct=0;tot=0;losses=0.
    for ix in range(batches):
        x,y=synthetic_batch(task,PROBE_BATCH,length,seed=992211+length*43+ix)
        logits=model(x)
        correct+=int((logits.argmax(-1)==y).sum().item())
        tot+=int(y.numel())
        losses+=float(F.cross_entropy(logits,y,reduction='sum').item())
    model.train()
    return {'accuracy_pct':100*correct/tot,'cross_entropy':losses/tot,'val_samples':tot}




In [ ]:

PROBE_ROWS=[]; PROBE_CURVES=[]
if RUN_PROBES:
    for task,classes in [('state_mod4',4),('delayed_recall',6)]:
        for seed in PROBE_SEEDS:
            states=snapshot_pair(lambda order:PPLSTMProbeModel(classes,order=order),seed)
            for order in ORDERS:
                model=PPLSTMProbeModel(classes,order=order).to(DEVICE)
                model.load_state_dict(states[order])
                opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=.01)
                x0,y0=synthetic_batch(task,PROBE_BATCH,PROBE_TRAIN_CONTEXT,seed+10000)
                backend,graph=start_backend(model,x0,y0)
                for step in range(PROBE_STEPS):
                    x,y=synthetic_batch(task,PROBE_BATCH,PROBE_TRAIN_CONTEXT,seed+10000+step)
                    update(model,opt,backend,graph,x,y,1e-3)
                    if (step+1)%max(1,PROBE_STEPS//5)==0:
                        stats=eval_probe(model,task,PROBE_TRAIN_CONTEXT,batches=8)
                        PROBE_CURVES.append(dict(task=task,seed=seed,order=order,
                                                 step=step+1,accuracy_pct=stats['accuracy_pct']))
                        save_csv(OUT/'order_probe_curves.csv',PROBE_CURVES)
                for length in PROBE_EVAL_CONTEXTS:
                    stats=eval_probe(model,task,length,batches=12)
                    rec=dict(task=task,seed=seed,order=order,eval_context=length,
                             train_context=PROBE_TRAIN_CONTEXT,steps=PROBE_STEPS,
                             accuracy_pct=stats['accuracy_pct'],cross_entropy=stats['cross_entropy'],
                             examples=stats['val_samples'],chance_accuracy_pct=100/classes,
                             parameters=count_params(model),backend=backend)
                    PROBE_ROWS.append(rec);save_csv(OUT/'order_probe_final.csv',PROBE_ROWS)
                    print('PROBE',rec)
                del model,opt,graph
                torch.cuda.empty_cache();gc.collect()


## CSV and ZIP results

In [ ]:

report={'trained_in_chat':False,
        'protocol':{'A':'mLSTM→PPLSTM→mLSTM→PPLSTM','B':'PPLSTM→mLSTM→PPLSTM→mLSTM',
                    'matching':'Copied weights of like blocks, embeddings, norms and head',
                    'language_tokens_per_model':TARGET_TOKENS,'language_seeds':ABLATION_SEEDS,
                    'memory_seeds':PROBE_SEEDS,'memory_steps':PROBE_STEPS,
                    'precision':'fp32','device':torch.cuda.get_device_name(DEVICE),
                    'dataset':DATA_INFO,'warning':'One seed is exploratory, not statistically conclusive'},
        'language_results':LM_FINAL,'memory_results':PROBE_ROWS}
safe_json_write(OUT/'order_ablation_summary.json',report)
archive=export_metric_bundle()
print('Results ZIP:',archive)
print('LM:');display(pd.DataFrame(LM_FINAL))
if PROBE_ROWS:
    print('Separate trained memory tasks:');display(pd.DataFrame(PROBE_ROWS))
